# 01. KBO 투구 데이터 탐색

원본: [slothman3878/kbo_playbyplay](https://huggingface.co/datasets/slothman3878/kbo_playbyplay) (CC BY 4.0)

확인할 것
1. 컬럼 구성과 `pitch_result` / `events` 분포
2. 반 이닝 하나의 투구 흐름 (B-S-O)
3. 아웃카운트가 타석 경계에서만 바뀌는지
4. 반 이닝당 투구 수 → 오디오 길이 추정

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 60)

# 가장 작은 2026 시즌 파일만 받아서 본다
URL = "https://huggingface.co/datasets/slothman3878/kbo_playbyplay/resolve/main/v0/kbo_pbp_2026.parquet"
df = pd.read_parquet(URL)

print(df.shape)
df.head()

(144779, 48)


,game_pk,game_date,home_team,away_team,inning,inning_topbot,at_bat_number,pitch_number,batter,pitcher,batter_name,pitcher_name,balls,strikes,outs_when_up,on_1b,on_2b,on_3b,home_score,away_score,pitch_result,type,pitch_type,pitch_name,_naver_pitch_name,release_speed_kmh,plate_x,sz_top,sz_bot,release_pos_x,release_pos_y,release_pos_z,vx0,vy0,vz0,ax,ay,az,stand,plate_z,events,post_home_score,post_away_score,post_outs,runs_scored,post_on_1b,post_on_2b,post_on_3b
0,20260328HTSK02026,2026-03-28,SK,HT,1,top,1,1,65653,55855,김호령,화이트,0,0,0,None,None,None,0,0,B,B,FF,4-Seam Fastball,직구,151.0,-0.815332,3.311,1.606,-1.84399,50.0,6.09243,4.31578,-136.156,-4.32629,-8.46753,33.9235,-9.55825,R,3.758225,walk,0,0,0,0,65653,None,None
1,20260328HTSK02026,2026-03-28,SK,HT,1,top,1,2,65653,55855,김호령,화이트,1,0,0,None,None,None,0,0,F,S,FC,Cutter,커터,144.0,0.235958,3.311,1.606,-1.73768,50.0,5.91235,5.55006,-130.117,-6.16607,-2.73322,25.1091,-20.91770,R,1.862381,walk,0,0,0,0,65653,None,None
2,20260328HTSK02026,2026-03-28,SK,HT,1,top,1,3,65653,55855,김호령,화이트,1,1,0,None,None,None,0,0,B,B,ST,Sweeper,스위퍼,135.0,1.760570,3.311,1.606,-1.75254,50.0,5.92856,6.43673,-122.627,-4.60085,8.83575,28.8198,-33.83750,R,0.955346,walk,0,0,0,0,65653,None,None
3,20260328HTSK02026,2026-03-28,SK,HT,1,top,1,4,65653,55855,김호령,화이트,2,1,0,None,None,None,0,0,S,S,FF,4-Seam Fastball,직구,151.0,-0.352905,3.311,1.606,-1.73510,50.0,5.96777,6.70145,-137.074,-5.00335,-16.09540,32.3440,-11.87370,R,3.244286,walk,0,0,0,0,65653,None,None
4,20260328HTSK02026,2026-03-28,SK,HT,1,top,1,5,65653,55855,김호령,화이트,2,2,0,None,None,None,0,0,B,B,FF,4-Seam Fastball,직구,152.0,-0.223044,3.311,1.606,-1.63417,50.0,6.12424,6.26962,-137.142,-4.04041,-13.33480,27.0299,-11.37000,R,3.825042,walk,0,0,0,0,65653,None,None


In [2]:
# 투구 결과 코드: B 볼 | T 스트라이크 | S 헛스윙 | F 파울 | V 번트헛스윙 | W 번트파울 | H 타격
print(df["pitch_result"].value_counts(dropna=False), "\n")

# 타석 결과 (타석 마지막 투구에만 값이 있을 것으로 예상)
print(df["events"].value_counts(dropna=False))

pitch_result
B       53497
T       26213
H       25902
F       24613
S       13968
W         458
V          77
None       51
Name: count, dtype: int64 

events
field_out          50088
strikeout          34555
single             21642
walk               19517
double              5034
fielders_choice     3816
home_run            2880
double_play         2137
hit_by_pitch        1623
field_error         1095
sac_fly              990
sac_bunt             686
triple               453
None                 258
triple_play            5
Name: count, dtype: int64


In [3]:
# 첫 경기 1회초를 투구 순서대로 출력
cols = ["at_bat_number", "pitch_number", "pitcher_name", "batter_name",
        "balls", "strikes", "outs_when_up", "pitch_result",
        "_naver_pitch_name", "release_speed_kmh", "events", "post_outs"]

game = df[df["game_pk"] == df["game_pk"].iloc[0]]
one = game[(game["inning"] == 1) & (game["inning_topbot"] == "top")]

print(game["game_pk"].iloc[0], game["away_team"].iloc[0], "@", game["home_team"].iloc[0])
one[cols]

20260328HTSK02026 HT @ SK


,at_bat_number,pitch_number,pitcher_name,batter_name,balls,strikes,outs_when_up,pitch_result,_naver_pitch_name,release_speed_kmh,events,post_outs
0,1,1,화이트,김호령,0,0,0,B,직구,151.0,walk,0
1,1,2,화이트,김호령,1,0,0,F,커터,144.0,walk,0
2,1,3,화이트,김호령,1,1,0,B,스위퍼,135.0,walk,0
3,1,4,화이트,김호령,2,1,0,S,직구,151.0,walk,0
4,1,5,화이트,김호령,2,2,0,B,직구,152.0,walk,0
5,1,6,화이트,김호령,3,2,0,F,직구,151.0,walk,0
6,1,7,화이트,김호령,3,2,0,F,직구,152.0,walk,0
7,1,8,화이트,김호령,3,2,0,B,커터,144.0,walk,0
8,2,1,화이트,카스트로,0,0,0,B,직구,149.0,double,0
9,2,2,화이트,카스트로,1,0,0,B,커브,126.0,double,0


In [4]:
# 아웃카운트 검증: 다음 타석의 outs_when_up == 이번 타석의 post_outs 인가?
HALF = ["game_pk", "inning", "inning_topbot"]

# 타석당 마지막 투구 한 줄 (행은 경기 진행 순서로 정렬돼 있음)
pa = df.groupby(["game_pk", "at_bat_number"], sort=False).tail(1).copy()
pa["next_outs"] = pa.groupby(HALF, sort=False)["outs_when_up"].shift(-1)

first_pa = pa.groupby(HALF, sort=False).head(1)
mismatch = pa[pa["next_outs"].notna() & (pa["next_outs"] != pa["post_outs"])]

print("타석 수:", len(pa))
print("반 이닝 첫 타석인데 아웃 != 0:", (first_pa["outs_when_up"] != 0).sum())
print("타석 사이 아웃 불일치:", len(mismatch))
mismatch[["game_pk", "inning", "inning_topbot", "at_bat_number", "events", "post_outs", "next_outs"]].head(10)

타석 수: 37192
반 이닝 첫 타석인데 아웃 != 0: 0
타석 사이 아웃 불일치: 0


,game_pk,inning,inning_topbot,at_bat_number,events,post_outs,next_outs


In [5]:
# 반 이닝당 투구 수 분포
pitches = df[df["pitch_number"] > 0]  # 고의4구(투구 없음) 행 제외
per_half = pitches.groupby(HALF, sort=False).size()
print(per_half.describe(), "\n")

# 오디오 길이 추정: 투구 1개당 중계 멘트 약 15초로 가정
SEC_PER_PITCH = 15
print(f"반 이닝 평균 약 {per_half.mean() * SEC_PER_PITCH / 60:.1f}분")
print(f"반 이닝 400개(train 300 + dev 50 + test 50) 약 {per_half.mean() * SEC_PER_PITCH * 400 / 3600:.1f}시간")

count    8352.000000
mean       17.328544
std         7.137650
min         3.000000
25%        12.000000
50%        16.000000
75%        21.000000
max        61.000000
dtype: float64 

반 이닝 평균 약 4.3분
반 이닝 400개(train 300 + dev 50 + test 50) 약 28.9시간
